# Введение в MapReduce модель на Python


In [ ]:
from typing import NamedTuple # requires python 3.6+
from typing import Iterator

In [ ]:
def MAP(_, row:NamedTuple):
  if (row.gender == 'female'):
    yield (row.age, row)

def REDUCE(age:str, rows:Iterator[NamedTuple]):
  sum = 0
  count = 0
  for row in rows:
    sum += row.social_contacts
    count += 1
  if (count > 0):
    yield (age, sum/count)
  else:
    yield (age, 0)

Модель элемента данных

In [ ]:
class User(NamedTuple):
  id: int
  age: str
  social_contacts: int
  gender: str

In [ ]:
input_collection = [
    User(id=0, age=55, gender='male', social_contacts=20),
    User(id=1, age=25, gender='female', social_contacts=240),
    User(id=2, age=25, gender='female', social_contacts=500),
    User(id=3, age=33, gender='female', social_contacts=800)
]

Функция RECORDREADER моделирует чтение элементов с диска или по сети.

In [ ]:
def RECORDREADER():
  return [(u.id, u) for u in input_collection]

In [ ]:
list(RECORDREADER())

[(0, User(id=0, age=55, social_contacts=20, gender='male')),
 (1, User(id=1, age=25, social_contacts=240, gender='female')),
 (2, User(id=2, age=25, social_contacts=500, gender='female')),
 (3, User(id=3, age=33, social_contacts=800, gender='female'))]

In [ ]:
def flatten(nested_iterable):
  for iterable in nested_iterable:
    for element in iterable:
      yield element

In [ ]:
map_output = flatten(map(lambda x: MAP(*x), RECORDREADER()))
map_output = list(map_output) # materialize
map_output

[(25, User(id=1, age=25, social_contacts=240, gender='female')),
 (25, User(id=2, age=25, social_contacts=500, gender='female')),
 (33, User(id=3, age=33, social_contacts=800, gender='female'))]

In [ ]:
def groupbykey(iterable):
  t = {}
  for (k2, v2) in iterable:
    t[k2] = t.get(k2, []) + [v2]
  return t.items()

In [ ]:
shuffle_output = groupbykey(map_output)
shuffle_output = list(shuffle_output)
shuffle_output

[(25,
  [User(id=1, age=25, social_contacts=240, gender='female'),
   User(id=2, age=25, social_contacts=500, gender='female')]),
 (33, [User(id=3, age=33, social_contacts=800, gender='female')])]

In [ ]:
reduce_output = flatten(map(lambda x: REDUCE(*x), shuffle_output))
reduce_output = list(reduce_output)
reduce_output

[(25, 370.0), (33, 800.0)]

Все действия одним конвейером!

In [ ]:
list(flatten(map(lambda x: REDUCE(*x), groupbykey(flatten(map(lambda x: MAP(*x), RECORDREADER()))))))

[(25, 370.0), (33, 800.0)]

# **MapReduce**
Выделим общую для всех пользователей часть системы в отдельную функцию высшего порядка. Это наиболее простая модель MapReduce, без учёта распределённого хранения данных.

Пользователь для решения своей задачи реализует RECORDREADER, MAP, REDUCE.

In [ ]:
def flatten(nested_iterable):
  for iterable in nested_iterable:
    for element in iterable:
      yield element

def groupbykey(iterable):
  t = {}
  for (k2, v2) in iterable:
    t[k2] = t.get(k2, []) + [v2]
  return t.items()

def MapReduce(RECORDREADER, MAP, REDUCE):
  return flatten(map(lambda x: REDUCE(*x), groupbykey(flatten(map(lambda x: MAP(*x), RECORDREADER())))))

## Спецификация MapReduce



```
f (k1, v1) -> (k2,v2)*
g (k2, v2*) -> (k3,v3)*

mapreduce ((k1,v1)*) -> (k3,v3)*
groupby ((k2,v2)*) -> (k2,v2*)*
flatten (e2**) -> e2*

mapreduce .map(f).flatten.groupby(k2).map(g).flatten
```




# Примеры

## SQL

In [ ]:
from typing import NamedTuple # requires python 3.6+
from typing import Iterator

class User(NamedTuple):
  id: int
  age: str
  social_contacts: int
  gender: str

input_collection = [
    User(id=0, age=55, gender='male', social_contacts=20),
    User(id=1, age=25, gender='female', social_contacts=240),
    User(id=2, age=25, gender='female', social_contacts=500),
    User(id=3, age=33, gender='female', social_contacts=800)
]

def MAP(_, row:NamedTuple):
  if (row.gender == 'female'):
    yield (row.age, row)

def REDUCE(age:str, rows:Iterator[NamedTuple]):
  sum = 0
  count = 0
  for row in rows:
    sum += row.social_contacts
    count += 1
  if (count > 0):
    yield (age, sum/count)
  else:
    yield (age, 0)

def RECORDREADER():
  return [(u.id, u) for u in input_collection]

output = MapReduce(RECORDREADER, MAP, REDUCE)
output = list(output)
output

[(25, 370.0), (33, 800.0)]

## Matrix-Vector multiplication

In [ ]:
from typing import Iterator
import numpy as np

mat = np.ones((5,4))
vec = np.random.rand(4) # in-memory vector in all map tasks

def MAP(coordinates:(int, int), value:int):
  i, j = coordinates
  yield (i, value*vec[j])

def REDUCE(i:int, products:Iterator[NamedTuple]):
  sum = 0
  for p in products:
    sum += p
  yield (i, sum)

def RECORDREADER():
  for i in range(mat.shape[0]):
    for j in range(mat.shape[1]):
      yield ((i, j), mat[i,j])

output = MapReduce(RECORDREADER, MAP, REDUCE)
output = list(output)
output

[(0, np.float64(2.105039491218208)),
 (1, np.float64(2.105039491218208)),
 (2, np.float64(2.105039491218208)),
 (3, np.float64(2.105039491218208)),
 (4, np.float64(2.105039491218208))]

## Inverted index

In [ ]:
from typing import Iterator

d1 = "it is what it is"
d2 = "what is it"
d3 = "it is a banana"
documents = [d1, d2, d3]

def RECORDREADER():
  for (docid, document) in enumerate(documents):
    yield ("{}".format(docid), document)

def MAP(docId:str, body:str):
  for word in set(body.split(' ')):
    yield (word, docId)

def REDUCE(word:str, docIds:Iterator[str]):
  yield (word, sorted(docIds))

output = MapReduce(RECORDREADER, MAP, REDUCE)
output = list(output)
output

[('is', ['0', '1', '2']),
 ('it', ['0', '1', '2']),
 ('what', ['0', '1']),
 ('a', ['2']),
 ('banana', ['2'])]

## WordCount

In [ ]:
from typing import Iterator

d1 = """
it is what it is
it is what it is
it is what it is"""
d2 = """
what is it
what is it"""
d3 = """
it is a banana"""
documents = [d1, d2, d3]

def RECORDREADER():
  for (docid, document) in enumerate(documents):
    for (lineid, line) in enumerate(document.split('\n')):
      yield ("{}:{}".format(docid,lineid), line)

def MAP(docId:str, line:str):
  for word in line.split(" "):
    yield (word, 1)

def REDUCE(word:str, counts:Iterator[int]):
  sum = 0
  for c in counts:
    sum += c
  yield (word, sum)

output = MapReduce(RECORDREADER, MAP, REDUCE)
output = list(output)
output

[('', 3), ('it', 9), ('is', 9), ('what', 5), ('a', 1), ('banana', 1)]

# MapReduce Distributed

Добавляется в модель фабрика RECORDREARER-ов --- INPUTFORMAT, функция распределения промежуточных результатов по партициям PARTITIONER, и функция COMBINER для частичной аггрегации промежуточных результатов до распределения по новым партициям.

In [ ]:
def flatten(nested_iterable):
  for iterable in nested_iterable:
    for element in iterable:
      yield element

def groupbykey(iterable):
  t = {}
  for (k2, v2) in iterable:
    t[k2] = t.get(k2, []) + [v2]
  return t.items()

def groupbykey_distributed(map_partitions, PARTITIONER):
  global reducers
  partitions = [dict() for _ in range(reducers)]
  for map_partition in map_partitions:
    for (k2, v2) in map_partition:
      p = partitions[PARTITIONER(k2)]
      p[k2] = p.get(k2, []) + [v2]
  return [(partition_id, sorted(partition.items(), key=lambda x: x[0])) for (partition_id, partition) in enumerate(partitions)]

def PARTITIONER(obj):
  global reducers
  return hash(obj) % reducers

def MapReduceDistributed(INPUTFORMAT, MAP, REDUCE, PARTITIONER=PARTITIONER, COMBINER=None):
  map_partitions = map(lambda record_reader: flatten(map(lambda k1v1: MAP(*k1v1), record_reader)), INPUTFORMAT())
  if COMBINER != None:
    map_partitions = map(lambda map_partition: flatten(map(lambda k2v2: COMBINER(*k2v2), groupbykey(map_partition))), map_partitions)
  reduce_partitions = groupbykey_distributed(map_partitions, PARTITIONER) # shuffle
  reduce_outputs = map(lambda reduce_partition: (reduce_partition[0], flatten(map(lambda reduce_input_group: REDUCE(*reduce_input_group), reduce_partition[1]))), reduce_partitions)

  print("{} key-value pairs were sent over a network.".format(sum([len(vs) for (k,vs) in flatten([partition for (partition_id, partition) in reduce_partitions])])))
  return reduce_outputs

## Спецификация MapReduce Distributed


```
f (k1, v1) -> (k2,v2)*
g (k2, v2*) -> (k3,v3)*

e1 (k1, v1)
e2 (k2, v2)
partition1 (k2, v2)*
partition2 (k2, v2*)*

flatmap (e1->e2*, e1*) -> partition1*
groupby (partition1*) -> partition2*

mapreduce ((k1,v1)*) -> (k3,v3)*
mapreduce .flatmap(f).groupby(k2).flatmap(g)
```



## WordCount

In [ ]:
from typing import Iterator
import numpy as np

d1 = """
it is what it is
it is what it is
it is what it is"""
d2 = """
what is it
what is it"""
d3 = """
it is a banana"""
documents = [d1, d2, d3, d1, d2, d3]

maps = 3
reducers = 2

def INPUTFORMAT():
  global maps

  def RECORDREADER(split):
    for (docid, document) in enumerate(split):
      for (lineid, line) in enumerate(document.split('\n')):
        yield ("{}:{}".format(docid,lineid), line)

  split_size =  int(np.ceil(len(documents)/maps))
  for i in range(0, len(documents), split_size):
    yield RECORDREADER(documents[i:i+split_size])

def MAP(docId:str, line:str):
  for word in line.split(" "):
    yield (word, 1)

def REDUCE(word:str, counts:Iterator[int]):
  sum = 0
  for c in counts:
    sum += c
  yield (word, sum)

# try to set COMBINER=REDUCER and look at the number of values sent over the network
partitioned_output = MapReduceDistributed(INPUTFORMAT, MAP, REDUCE, COMBINER=None)
partitioned_output = [(partition_id, list(partition)) for (partition_id, partition) in partitioned_output]
partitioned_output

56 key-value pairs were sent over a network.


[(0, [('', 6), ('banana', 2), ('is', 18)]),
 (1, [('a', 2), ('it', 18), ('what', 10)])]

## TeraSort

In [ ]:
import numpy as np

input_values = np.random.rand(30)
maps = 3
reducers = 2
min_value = 0.0
max_value = 1.0

def INPUTFORMAT():
  global maps

  def RECORDREADER(split):
    for value in split:
        yield (value, None)

  split_size =  int(np.ceil(len(input_values)/maps))
  for i in range(0, len(input_values), split_size):
    yield RECORDREADER(input_values[i:i+split_size])

def MAP(value:int, _):
  yield (value, None)

def PARTITIONER(key):
  global reducers
  global max_value
  global min_value
  bucket_size = (max_value-min_value)/reducers
  bucket_id = 0
  while((key>(bucket_id+1)*bucket_size) and ((bucket_id+1)*bucket_size<max_value)):
    bucket_id += 1
  return bucket_id

def REDUCE(value:int, _):
  yield (None,value)

partitioned_output = MapReduceDistributed(INPUTFORMAT, MAP, REDUCE, COMBINER=None, PARTITIONER=PARTITIONER)
partitioned_output = [(partition_id, list(partition)) for (partition_id, partition) in partitioned_output]
partitioned_output

30 key-value pairs were sent over a network.


[(0,
  [(None, np.float64(0.013792152323922502)),
   (None, np.float64(0.05464430517532615)),
   (None, np.float64(0.08258651781770088)),
   (None, np.float64(0.10177909433261134)),
   (None, np.float64(0.12094171423752509)),
   (None, np.float64(0.20187126750230178)),
   (None, np.float64(0.2025758390559269)),
   (None, np.float64(0.2663917985613342)),
   (None, np.float64(0.35073006622427627)),
   (None, np.float64(0.3853686227003491)),
   (None, np.float64(0.40020838145550086)),
   (None, np.float64(0.4271548270366198)),
   (None, np.float64(0.4352287610626546))]),
 (1,
  [(None, np.float64(0.5004774217950984)),
   (None, np.float64(0.5317836475491994)),
   (None, np.float64(0.538204089295316)),
   (None, np.float64(0.6130109421979479)),
   (None, np.float64(0.7207674421381977)),
   (None, np.float64(0.722439081224504)),
   (None, np.float64(0.7380345793342502)),
   (None, np.float64(0.7452276790815987)),
   (None, np.float64(0.7485800657495661)),
   (None, np.float64(0.861025124148

# Упражнения
Упражнения взяты из Rajaraman A., Ullman J. D. Mining of massive datasets. – Cambridge University Press, 2011.


Для выполнения заданий переопределите функции RECORDREADER, MAP, REDUCE. Для модели распределённой системы может потребоваться переопределение функций PARTITION и COMBINER.

### Максимальное значение ряда

Разработайте MapReduce алгоритм, который находит максимальное число входного списка чисел.

In [ ]:
input_numbers = [3, 7, 2, 9, 1, 5, 8, 4, 6]

def RECORDREADER():
  for (i, num) in enumerate(input_numbers):
    yield (i, num)

def MAP(key, value):
  yield ("max", value)

def REDUCE(key, values):
  yield (key, max(values))

output = list(MapReduce(RECORDREADER, MAP, REDUCE))
print(output)
assert output == [("max", 9)]

[('max', 9)]


### Арифметическое среднее

Разработайте MapReduce алгоритм, который находит арифметическое среднее.

$$\overline{X} = \frac{1}{n}\sum_{i=0}^{n} x_i$$


In [ ]:
input_numbers = [10, 20, 30, 40, 50]

def RECORDREADER():
  for (i, num) in enumerate(input_numbers):
    yield (i, num)

def MAP(key, value):
  yield ("mean", (value, 1))

def REDUCE(key, values):
  total = sum(v for v, c in values)
  count = sum(c for v, c in values)
  yield (key, total / count)

output = list(MapReduce(RECORDREADER, MAP, REDUCE))
print(output)
assert output == [("mean", 30.0)]

[('mean', 30.0)]


### GroupByKey на основе сортировки

Реализуйте groupByKey на основе сортировки, проверьте его работу на примерах

In [ ]:
def groupbykey_sort(iterable):
  sorted_list = sorted(iterable, key=lambda x: x[0])
  result = {}
  for (k, v) in sorted_list:
    result[k] = result.get(k, []) + [v]
  return result.items()

test_data = [(1, 'a'), (2, 'b'), (1, 'c'), (3, 'd'), (2, 'e')]
print(list(groupbykey_sort(test_data)))

[(1, ['a', 'c']), (2, ['b', 'e']), (3, ['d'])]


### Drop duplicates (set construction, unique elements, distinct)

Реализуйте распределённую операцию исключения дубликатов

In [ ]:
input_data = [1, 2, 3, 2, 1, 4, 3, 5]

def RECORDREADER():
  for (i, val) in enumerate(input_data):
    yield (i, val)

def MAP(key, value):
  yield (value, None)

def REDUCE(key, values):
  yield (key, key)

output = list(MapReduce(RECORDREADER, MAP, REDUCE))
print(output)

[(1, 1), (2, 2), (3, 3), (4, 4), (5, 5)]


#Операторы реляционной алгебры
### Selection (Выборка)

**The Map Function**: Для  каждого кортежа $t \in R$ вычисляется истинность предиката $C$. В случае истины создаётся пара ключ-значение $(t, t)$. В паре ключ и значение одинаковы, равны $t$.

**The Reduce Function:** Роль функции Reduce выполняет функция идентичности, которая возвращает то же значение, что получила на вход.



In [ ]:
class User(NamedTuple):
  id: int
  age: int
  social_contacts: int
  gender: str

R = [
  User(id=0, age=55, gender='male', social_contacts=20),
  User(id=1, age=25, gender='female', social_contacts=240),
  User(id=2, age=25, gender='female', social_contacts=500),
  User(id=3, age=33, gender='female', social_contacts=800)
]

C = lambda t: t.gender == 'female'

def RECORDREADER():
  for t in R:
    yield (t, t)

def MAP(key, t):
  if C(t):
    yield (t, t)

def REDUCE(key, values):
  yield (key, values[0])

output = list(MapReduce(RECORDREADER, MAP, REDUCE))
print(output)

[(User(id=1, age=25, social_contacts=240, gender='female'), User(id=1, age=25, social_contacts=240, gender='female')), (User(id=2, age=25, social_contacts=500, gender='female'), User(id=2, age=25, social_contacts=500, gender='female')), (User(id=3, age=33, social_contacts=800, gender='female'), User(id=3, age=33, social_contacts=800, gender='female'))]


### Projection (Проекция)

Проекция на множество атрибутов $S$.

**The Map Function:** Для каждого кортежа $t \in R$ создайте кортеж $t′$, исключая  из $t$ те значения, атрибуты которых не принадлежат  $S$. Верните пару $(t′, t′)$.

**The Reduce Function:** Для каждого ключа $t′$, созданного любой Map задачей, вы получаете одну или несколько пар $(t′, t′)$. Reduce функция преобразует $(t′, [t′, t′, . . . , t′])$ в $(t′, t′)$, так, что для ключа $t′$ возвращается одна пара  $(t′, t′)$.

In [ ]:
S = {'age', 'gender'}

def RECORDREADER():
  for t in R:
    yield (t, t)

def MAP(key, t):
  t_prime = (t.age, t.gender)
  yield (t_prime, t_prime)

def REDUCE(key, values):
  yield (key, key)

output = list(MapReduce(RECORDREADER, MAP, REDUCE))
print(output)

[((55, 'male'), (55, 'male')), ((25, 'female'), (25, 'female')), ((33, 'female'), (33, 'female'))]


### Union (Объединение)

**The Map Function:** Превратите каждый входной кортеж $t$ в пару ключ-значение $(t, t)$.

**The Reduce Function:** С каждым ключом $t$ будет ассоциировано одно или два значений. В обоих случаях создайте $(t, t)$ в качестве выходного значения.

In [ ]:
R_set = [1, 2, 3, 4]
S_set = [3, 4, 5, 6]

def RECORDREADER():
  for t in R_set:
    yield (t, t)
  for t in S_set:
    yield (t, t)

def MAP(key, value):
  yield (value, value)

def REDUCE(key, values):
  yield (key, key)

output = list(MapReduce(RECORDREADER, MAP, REDUCE))
print(output)

[(1, 1), (2, 2), (3, 3), (4, 4), (5, 5), (6, 6)]


### Intersection (Пересечение)

**The Map Function:** Превратите каждый кортеж $t$ в пары ключ-значение $(t, t)$.

**The Reduce Function:** Если для ключа $t$ есть список из двух элементов $[t, t]$ $-$ создайте пару $(t, t)$. Иначе, ничего не создавайте.

In [ ]:
def RECORDREADER():
  for t in R_set:
    yield (t, t)
  for t in S_set:
    yield (t, t)

def MAP(key, value):
  yield (value, value)

def REDUCE(key, values):
  if len(values) == 2:
    yield (key, key)

output = list(MapReduce(RECORDREADER, MAP, REDUCE))
print(output)

[(3, 3), (4, 4)]


### Difference (Разница)

**The Map Function:** Для кортежа $t \in R$, создайте пару $(t, R)$, и для кортежа $t \in S$, создайте пару $(t, S)$. Задумка заключается в том, чтобы значение пары было именем отношения $R$ or $S$, которому принадлежит кортеж (а лучше, единичный бит, по которому можно два отношения различить $R$ or $S$), а не весь набор атрибутов отношения.

**The Reduce Function:** Для каждого ключа $t$, если соответствующее значение является списком $[R]$, создайте пару $(t, t)$. В иных случаях не предпринимайте действий.

In [ ]:
def RECORDREADER():
  for t in R_set:
    yield (t, ('R', t))
  for t in S_set:
    yield (t, ('S', t))

def MAP(key, value):
  yield (value[1], value[0])

def REDUCE(key, values):
  if values == ['R']:
    yield (key, key)

output = list(MapReduce(RECORDREADER, MAP, REDUCE))
print(output)

[(1, 1), (2, 2)]


### Natural Join

**The Map Function:** Для каждого кортежа $(a, b)$ отношения $R$, создайте пару $(b,(R, a))$. Для каждого кортежа $(b, c)$ отношения $S$, создайте пару $(b,(S, c))$.

**The Reduce Function:** Каждый ключ $b$ будет асоциирован со списком пар, которые принимают форму либо $(R, a)$, либо $(S, c)$. Создайте все пары, одни, состоящие из  первого компонента $R$, а другие, из первого компонента $S$, то есть $(R, a)$ и $(S, c)$. На выходе вы получаете последовательность пар ключ-значение из списков ключей и значений. Ключ не нужен. Каждое значение, это тройка $(a, b, c)$ такая, что $(R, a)$ и $(S, c)$ это принадлежат входному списку значений.

In [ ]:
R_rel = [(1, 2), (3, 4), (5, 4)]
S_rel = [(2, 6), (4, 8), (4, 9)]

def RECORDREADER():
  for (a, b) in R_rel:
    yield ((a, b), ('R', a, b))
  for (b, c) in S_rel:
    yield ((b, c), ('S', b, c))

def MAP(key, value):
  if value[0] == 'R':
    a, b = value[1], value[2]
    yield (b, ('R', a))
  else:
    b, c = value[1], value[2]
    yield (b, ('S', c))

def REDUCE(b, values):
  r_vals = [a for (tag, a) in values if tag == 'R']
  s_vals = [c for (tag, c) in values if tag == 'S']
  for a in r_vals:
    for c in s_vals:
      yield (None, (a, b, c))

output = list(MapReduce(RECORDREADER, MAP, REDUCE))
print(output)

[(None, (1, 2, 6)), (None, (3, 4, 8)), (None, (3, 4, 9)), (None, (5, 4, 8)), (None, (5, 4, 9))]


### Grouping and Aggregation (Группировка и аггрегация)

**The Map Function:** Для каждого кортежа $(a, b, c$) создайте пару $(a, b)$.

**The Reduce Function:** Ключ представляет ту или иную группу. Примение аггрегирующую операцию $\theta$ к списку значений $[b1, b2, . . . , bn]$ ассоциированных с ключом $a$. Возвращайте в выходной поток $(a, x)$, где $x$ результат применения  $\theta$ к списку. Например, если $\theta$ это $SUM$, тогда $x = b1 + b2 + · · · + bn$, а если $\theta$ is $MAX$, тогда $x$ это максимальное из значений $b1, b2, . . . , bn$.

In [ ]:
tuples = [(1, 10, 'x'), (1, 20, 'y'), (2, 30, 'z'), (2, 40, 'w')]

def RECORDREADER():
  for (i, t) in enumerate(tuples):
    yield (i, t)

def MAP(key, t):
  a, b, c = t
  yield (a, b)

def REDUCE(a, values):
  yield (a, sum(values))

output = list(MapReduce(RECORDREADER, MAP, REDUCE))
print(output)

[(1, 30), (2, 70)]


#

### Matrix-Vector multiplication

Случай, когда вектор не помещается в памяти Map задачи


In [ ]:
mat = np.ones((5, 4))
vec = np.random.rand(4)

def RECORDREADER():
  for i in range(mat.shape[0]):
    for j in range(mat.shape[1]):
      yield (('M', i, j), mat[i, j])
  for j in range(len(vec)):
    yield (('V', j), vec[j])

def MAP(key, value):
  if key[0] == 'M':
    _, i, j = key
    yield (j, ('M', i, value))
  else:
    _, j = key
    yield (j, ('V', value))

def REDUCE(j, values):
  m_vals = []
  v_val = 0
  for val in values:
    if val[0] == 'M':
      m_vals.append((val[1], val[2]))
    else:
      v_val = val[1]
  for (i, m) in m_vals:
    yield ((i,), m * v_val)

output1 = list(MapReduce(RECORDREADER, MAP, REDUCE))

def RECORDREADER2():
  for item in output1:
    yield item

def MAP2(key, value):
  yield (key, value)

def REDUCE2(key, values):
  yield (key, sum(values))

output2 = list(MapReduce(RECORDREADER2, MAP2, REDUCE2))
print(output2)
ref = mat @ vec
print("Reference:", ref)

[((0,), np.float64(1.5416416899803589)), ((1,), np.float64(1.5416416899803589)), ((2,), np.float64(1.5416416899803589)), ((3,), np.float64(1.5416416899803589)), ((4,), np.float64(1.5416416899803589))]
Reference: [1.54164169 1.54164169 1.54164169 1.54164169 1.54164169]


## Matrix multiplication (Перемножение матриц)

Если у нас есть матрица $M$ с элементами $m_{ij}$ в строке $i$ и столбце $j$, и матрица $N$ с элементами $n_{jk}$ в строке $j$ и столбце $k$, тогда их произведение $P = MN$ есть матрица $P$ с элементами $p_{ik}$ в строке $i$ и столбце $k$, где

$$p_{ik} =\sum_{j} m_{ij}n_{jk}$$

Необходимым требованием является одинаковое количество столбцов в $M$ и строк в $N$, чтобы операция суммирования по  $j$ была осмысленной. Мы можем размышлять о матрице, как об отношении с тремя атрибутами: номер строки, номер столбца, само значение. Таким образом матрица $M$ предстваляется как отношение $ M(I, J, V )$, с кортежами $(i, j, m_{ij})$, и, аналогично, матрица $N$ представляется как отношение $N(J, K, W)$, с кортежами $(j, k, n_{jk})$. Так как большие матрицы как правило разреженные (большинство значений равно 0), и так как мы можем нулевыми значениями пренебречь (не хранить), такое реляционное представление достаточно эффективно для больших матриц. Однако, возможно, что координаты $i$, $j$, и $k$ неявно закодированы в смещение позиции элемента относительно начала файла, вместо явного хранения. Тогда, функция Map (или Reader) должна быть разработана таким образом, чтобы реконструировать компоненты $I$, $J$, и $K$ кортежей из смещения.

Произведение $MN$ это фактически join, за которым следуют группировка по ключу и аггрегация. Таким образом join отношений $M(I, J, V )$ и $N(J, K, W)$, имеющих общим только атрибут $J$, создаст кортежи $(i, j, k, v, w)$ из каждого кортежа $(i, j, v) \in M$ и кортежа $(j, k, w) \in N$. Такой 5 компонентный кортеж представляет пару элементов матрицы $(m_{ij} , n_{jk})$. Что нам хотелось бы получить на самом деле, это произведение этих элементов, то есть, 4 компонентный кортеж$(i, j, k, v \times w)$, так как он представляет произведение $m_{ij}n_{jk}$. Мы представляем отношение как результат одной MapReduce операции, в которой мы можем произвести группировку и аггрегацию, с $I$ и $K$  атрибутами, по которым идёт группировка, и суммой  $V \times W$.





In [ ]:
# MapReduce model
def flatten(nested_iterable):
  for iterable in nested_iterable:
    for element in iterable:
      yield element

def groupbykey(iterable):
  t = {}
  for (k2, v2) in iterable:
    t[k2] = t.get(k2, []) + [v2]
  return t.items()

def MapReduce(RECORDREADER, MAP, REDUCE):
  return flatten(map(lambda x: REDUCE(*x), groupbykey(flatten(map(lambda x: MAP(*x), RECORDREADER())))))

Реализуйте перемножение матриц с использованием модельного кода MapReduce для одной машины в случае, когда одна матрица хранится в памяти, а другая генерируется RECORDREADER-ом.

In [ ]:
import numpy as np

I = 2
J = 3
K = 4*10
small_mat = np.random.rand(I,J) # В памяти
big_mat = np.random.rand(J,K)   # Читается потоком

def RECORDREADER():
  for j in range(big_mat.shape[0]):
    for k in range(big_mat.shape[1]):
      yield ((j,k), big_mat[j,k])

def MAP(k1, v1):
  (j, k) = k1
  w = v1
  for i in range(small_mat.shape[0]):
    yield ((i, k), small_mat[i, j] * w)

def REDUCE(key, values):
  (i, k) = key
  yield (key, sum(values))

Проверьте своё решение

In [ ]:
# CHECK THE SOLUTION
reference_solution = np.matmul(small_mat, big_mat)
solution = MapReduce(RECORDREADER, MAP, REDUCE)

def asmatrix(reduce_output):
  reduce_output = list(reduce_output)
  I = max(i for ((i,k), vw) in reduce_output)+1
  K = max(k for ((i,k), vw) in reduce_output)+1
  mat = np.empty(shape=(I,K))
  for ((i,k), vw) in reduce_output:
    mat[i,k] = vw
  return mat

np.allclose(reference_solution, asmatrix(solution)) # should return true

True

In [ ]:
reduce_output = list(MapReduce(RECORDREADER, MAP, REDUCE))
max(i for ((i,k), vw) in reduce_output)

1

Реализуйте перемножение матриц  с использованием модельного кода MapReduce для одной машины в случае, когда обе матрицы генерируются в RECORDREADER. Например, сначала одна, а потом другая.

In [ ]:
I_dim = 2; J_dim = 3; K_dim = 4*10
small_mat = np.random.rand(I_dim, J_dim)
big_mat = np.random.rand(J_dim, K_dim)

def RECORDREADER():
  for j in range(big_mat.shape[0]):
    for k in range(big_mat.shape[1]):
      yield ((j, k), big_mat[j, k])

def MAP(k1, v1):
  (j, k) = k1
  w = v1
  for i in range(small_mat.shape[0]):
    yield ((i, k), small_mat[i, j] * w)

def REDUCE(key, values):
  (i, k) = key
  yield (key, sum(values))

def asmatrix(reduce_output):
  reduce_output = list(reduce_output)
  I = max(i for ((i,k), vw) in reduce_output)+1
  K = max(k for ((i,k), vw) in reduce_output)+1
  mat = np.empty(shape=(I,K))
  for ((i,k), vw) in reduce_output:
    mat[i,k] = vw
  return mat

reference_solution = np.matmul(small_mat, big_mat)
solution = MapReduce(RECORDREADER, MAP, REDUCE)
print("Matches:", np.allclose(reference_solution, asmatrix(solution)))

Matches: True


Реализуйте перемножение матриц с использованием модельного кода MapReduce Distributed, когда каждая матрица генерируется в своём RECORDREADER.

In [ ]:
M = np.random.rand(2, 3)
N = np.random.rand(3, 4)

def RECORDREADER():
  for i in range(M.shape[0]):
    for j in range(M.shape[1]):
      yield (('M', i, j), M[i, j])
  for j in range(N.shape[0]):
    for k in range(N.shape[1]):
      yield (('N', j, k), N[j, k])

def MAP(key, value):
  if key[0] == 'M':
    _, i, j = key
    for k in range(N.shape[1]):
      yield ((i, k), ('M', j, value))
  else:
    _, j, k = key
    for i in range(M.shape[0]):
      yield ((i, k), ('N', j, value))

def REDUCE(key, values):
  m_dict = {}
  n_dict = {}
  for val in values:
    if val[0] == 'M':
      m_dict[val[1]] = val[2]
    else:
      n_dict[val[1]] = val[2]
  s = sum(m_dict.get(j, 0) * n_dict.get(j, 0) for j in set(m_dict) | set(n_dict))
  yield (key, s)

ref = M @ N
sol = asmatrix(MapReduce(RECORDREADER, MAP, REDUCE))
print("Matches:", np.allclose(ref, sol))

Matches: True


Обобщите предыдущее решение на случай, когда каждая матрица генерируется несколькими RECORDREADER-ами, и проверьте его работоспособность. Будет ли работать решение, если RECORDREADER-ы будут генерировать случайное подмножество элементов матрицы?

In [ ]:
import numpy as np
from itertools import chain

M = np.random.rand(2, 3)
N = np.random.rand(3, 4)


def RECORDREADER_M_ROW_0():
    for j in range(M.shape[1]):
        yield (('M', 0, j), M[0, j])

def RECORDREADER_M_ROW_1():
    for j in range(M.shape[1]):
        yield (('M', 1, j), M[1, j])

def RECORDREADER_N_PART_1():
    for j in range(N.shape[0]):
        for k in range(2):
            yield (('N', j, k), N[j, k])

def RECORDREADER_N_PART_2():
    for j in range(N.shape[0]):
        for k in range(2, N.shape[1]):
            yield (('N', j, k), N[j, k])

def COMBINED_RECORDREADER():
    yield from RECORDREADER_M_ROW_0()
    yield from RECORDREADER_M_ROW_1()
    yield from RECORDREADER_N_PART_1()
    yield from RECORDREADER_N_PART_2()

def MAP(key, value):
    if key[0] == 'M':
        _, i, j = key
        for k in range(N.shape[1]):
            yield ((i, k), ('M', j, value))
    else:
        _, j, k = key
        for i in range(M.shape[0]):
            yield ((i, k), ('N', j, value))

def REDUCE(key, values):
    m_dict = {}
    n_dict = {}
    for val in values:
        if val[0] == 'M':
            m_dict[val[1]] = val[2]
        else:
            n_dict[val[1]] = val[2]
    s = sum(m_dict.get(j, 0) * n_dict.get(j, 0) for j in set(m_dict) | set(n_dict))
    yield (key, s)

ref = M @ N
sol = asmatrix(MapReduce(COMBINED_RECORDREADER, MAP, REDUCE))
print("Matches:", np.allclose(ref, sol))

Matches: True
